# OCR Gate B — full dev-subset-5 + emergency Vintern calibration 100

Prerequisites: (1) attach `thvu165/aic-2026-keyframes`; (2) attach either the human-evaluated `craft-gate-a-evaluation.json` or the original `ocr_gate_a_emergency_100_artifacts.zip`; (3) select one T4 GPU and enable Internet. Optionally attach a prior `ocr_gate_b_checkpoint.zip` to resume after a lost Kaggle session.

This notebook fails before model loading unless Gate A provides either a normal `PASS_THRESHOLD_SELECTED` or the exact `DEADLINE_OVERRIDE_KEEP_CURRENT`. It reruns all 4,164 dev frames with the selected CRAFT threshold, EasyOCR, router v2, and the pinned official Vintern FP16 revision. Only the human-review burden is reduced: exactly 100 candidates from 100 distinct frames, balanced 20/video and deterministically stratified. This is emergency single-annotator evidence, not a 300-frame calibration PASS. It never calls Gemini and never builds production SQLite.


In [ ]:
# Preserve Kaggle binary packages and expose only GPU 0.
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'; os.environ['TOKENIZERS_PARALLELISM']='false'
import importlib.metadata as md, importlib.util, subprocess, sys
def version_or_missing(name):
 try: return md.version(name)
 except md.PackageNotFoundError: return None
protected_names=['torch','torchvision','numpy','opencv-python','opencv-python-headless','scikit-image','pandas']
protected_before={name:version_or_missing(name) for name in protected_names}
packages=['transformers==4.43.4','tokenizers==0.19.1','huggingface-hub==0.24.7','easyocr==1.7.2','python-bidi==0.6.6','pyclipper==1.3.0.post6','ninja==1.11.1.4','sentencepiece==0.2.0']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir','--no-deps',*packages])
missing=[name for name in ['accelerate','safetensors','timm','einops','yaml','scipy','skimage','shapely'] if importlib.util.find_spec(name) is None]
assert not missing,('Kaggle image missing expected modules',missing)
protected_after={name:version_or_missing(name) for name in protected_names}
assert protected_before==protected_after,('pip changed Kaggle binary packages',protected_before,protected_after)
abi=subprocess.run([sys.executable,'-c','import numpy,cv2,pandas,skimage; print(numpy.__version__,cv2.__version__,pandas.__version__,skimage.__version__)'],text=True,capture_output=True)
assert abi.returncode==0,'Dirty binary ABI; stop and start a fresh Kaggle session.\n'+abi.stderr
print('ABI_PROBE',abi.stdout.strip(),{'protected':protected_after,'transformers':version_or_missing('transformers'),'easyocr':version_or_missing('easyocr')})


In [ ]:
from __future__ import annotations
import csv,gc,hashlib,json,math,re,shutil,time,unicodedata,urllib.request,zipfile
from collections import Counter,defaultdict
from datetime import datetime,timezone
from pathlib import Path
import cv2,numpy as np,torch
from PIL import Image,ImageDraw,ImageFont

DEV_EXPECTED={
 'L21_V001':{'count':1008,'uid_set_sha256':'d97f6d1cb014354b11942ea908b2f75fb7ec423dc44b87451b8fe157fc16eee2'},
 'L21_V002':{'count':843,'uid_set_sha256':'62773f4ecc75cac52d6df5f598734fd9f608afa3ea680a74da91b983031596ca'},
 'L21_V003':{'count':765,'uid_set_sha256':'b7d1fe33ecfeb644506b6015ee98a6e03b9b8a9baade275fab303cbe7bd4d03b'},
 'L21_V005':{'count':744,'uid_set_sha256':'2f16c1ddc51b87bfe3299f99910f01bf7e2b8ec718b6f99015cdda21982199b0'},
 'L21_V006':{'count':804,'uid_set_sha256':'b3b6e61ff01dbdef3afc2de96f8a077c992f43e32640264743b707bfe84942cc'},}
CRAFT_POLICY={'schema_version':2,'sample_frames':100,'video_ids':['L21_V001','L21_V002','L21_V003','L21_V005','L21_V006'],'frames_per_video':None,'sample_video_counts':{'L21_V001':60,'L21_V002':40,'L21_V003':0,'L21_V005':0,'L21_V006':0},'sample_uid_set_sha256':'4544608e596f9bb80d8356017ba2f5e5242717695a378566401b57d4dbb9a778','allow_current_fallback_for_gate_b':True,'evidence_limitations':['Emergency single-annotator review uses the already labeled first 100 deterministic Gate A rows: 60 L21_V001 and 40 L21_V002.','All 100 labeled frames contain text, so no-text false-positive rate is unavailable.','This deadline override may retain recall_current for full-dev Gate B but is not a balanced five-video threshold PASS.'],'min_region_recall':0.98,'min_text_frame_recall':0.99,'configs':[{'config_id':'recall_current','text_threshold':0.6,'low_text':0.3,'link_threshold':0.3},{'config_id':'balanced','text_threshold':0.7,'low_text':0.4,'link_threshold':0.4},{'config_id':'strict','text_threshold':0.8,'low_text':0.5,'link_threshold':0.5}]}
CRAFT_POLICY_SHA256='cb32f59bfacc22f8babbba65202d3dd40ce3d607c22817264e35b2a03bd9c561'
VINTERN_ROUTER_POLICY={'hard_confidence_threshold':0.4,'selective_confidence_ceiling':0.6,'noisy_character_ratio_threshold':0.34,'noisy_character_min_length':3,'max_candidate_fraction':0.4}
CALIBRATION_POLICY={'schema_version':2,'correctness_metric':'unicode_nfc_casefold_whitespace_exact_match','evidence_tier':'emergency_single_annotator_100','review_rows':100,'review_rows_per_video':20,'review_rows_per_stratum_per_video':4,'review_selection_seed':'vintern-gate-b-emergency-100-v1','min_ground_truth_frames':100,'min_total_labeled_regions':100,'min_bucket_samples':20,'allow_global_bucket_override':False,'output_length_upper_bounds':[4,12,32,96],'guard_margin_upper_bounds':[0.25,0.5,0.75],'mean_token_logprob_upper_bounds':[-2.0,-1.0,-0.5]}
CALIBRATION_POLICY_SHA256='03f98b31d37915f4759a87b829bb024f025829d6bf0baaba36f21f92286960fd'
MODEL_ID='5CD-AI/Vintern-1B-v3_5'; MODEL_REVISION='b98f263eab246eb5269ade64edbdca8a887dc44d'; MODEL_WEIGHT_BYTES=3752849256; MODEL_WEIGHT_SHA256='296a16a6bf28e6d3f0fb9298deba70b3cfa1d7519f4aa326e2f862bf2e63be05'
MODEL_REQUIRED_GIT_OIDS={'config.json':'2668519f652eddcc2abbb56a52518d38c4f88887','configuration_intern_vit.py':'7e630c456eb9cf350e55bf850c3ff72f445a7e17','configuration_internvl_chat.py':'799209432caf749e77de4c889ec03fe6a32fcdf9','conversation.py':'76fcea7f331de42b6aed7e39fdf80728f9784b7f','modeling_intern_vit.py':'1c5c043a4b860720b3b6e55107e8e6ecf0c573de','modeling_internvl_chat.py':'41f48cd5cb907e025725fc42ac819cf3f03c01b5'}
EASYOCR_WEIGHTS={'craft_mlt_25k':{'url':'https://github.com/JaidedAI/EasyOCR/releases/download/pre-v1.1.6/craft_mlt_25k.zip','zip_sha256':'8dc6a1c703a89ed56308ef742d26ebd45c656248cbbbda6e7fe60e569f873e65','weight_name':'craft_mlt_25k.pth','weight_sha256':'4a5efbfb48b4081100544e75e1e2b57f8de3d84f213004b14b85fd4b3748db17'},'latin_g2':{'url':'https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/latin_g2.zip','zip_sha256':'29f1920c493378da65a59793fb70e7e190504662b6bed57ab26f4067eb5f3769','weight_name':'latin_g2.pth','weight_sha256':'aaa95be1c4a9cb3496879bed7c520886ce1164f89e026f0c54488394e74e8c55'}}
INPUT_ROOT=Path('/kaggle/input'); WORK=Path('/kaggle/working/ocr-gate-b-dev5-emergency100-v2'); MODEL_DIR=WORK/'easyocr-models'; CROP_DIR=WORK/'region-crops'; REVIEW_DIR=WORK/'vintern-ground-truth-review'
EASY_JSONL=WORK/'easyocr-frames.jsonl'; CANDIDATE_JSONL=WORK/'vintern-candidates.jsonl'; VINTERN_JSONL=WORK/'vintern-results.jsonl'; STATE_JSON=WORK/'run-signature.json'; SELECTION_JSON=WORK/'vintern-calibration-selection.json'
CHECKPOINT_ZIP=Path('/kaggle/working/ocr_gate_b_checkpoint.zip'); REVIEW_CSV=WORK/'vintern-calibration-ground-truth.csv'; REVIEW_ZIP=Path('/kaggle/working/ocr_gate_b_ground_truth_review.zip'); REPORT_JSON=Path('/kaggle/working/ocr_gate_b_dev5_report.json')
for path in (WORK,MODEL_DIR,CROP_DIR,REVIEW_DIR): path.mkdir(parents=True,exist_ok=True)
def canonical_json(value): return json.dumps(value,sort_keys=True,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(canonical_json(CRAFT_POLICY)).hexdigest()==CRAFT_POLICY_SHA256
assert hashlib.sha256(canonical_json(CALIBRATION_POLICY)).hexdigest()==CALIBRATION_POLICY_SHA256
def sha256_file(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as h:
  for block in iter(lambda:h.read(8*1024*1024),b''): d.update(block)
 return d.hexdigest()
def git_blob_oid(path):
 path=Path(path); d=hashlib.sha1(); d.update(f'blob {path.stat().st_size}\0'.encode())
 with path.open('rb') as h:
  for block in iter(lambda:h.read(8*1024*1024),b''): d.update(block)
 return d.hexdigest()
def make_uid(video_id,shot_id,local_idx): return int.from_bytes(hashlib.blake2b(f'{video_id}:{shot_id}:{local_idx}'.encode(),digest_size=8).digest(),'big')>>1
def uid_set_sha256(values): return hashlib.sha256(''.join(f'{v}\n' for v in sorted(values)).encode()).hexdigest()
def stable_id(*parts): return hashlib.sha256('|'.join(map(str,parts)).encode()).hexdigest()[:24]
def append_jsonl(path,row):
 with Path(path).open('a',encoding='utf-8') as h: h.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n'); h.flush(); os.fsync(h.fileno())
def rewrite_jsonl(path,rows):
 temp=Path(str(path)+'.tmp')
 with temp.open('w',encoding='utf-8') as h:
  for row in rows: h.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+'\n')
 os.replace(temp,path)
def load_jsonl(path,id_field):
 rows=[]; seen=set(); path=Path(path)
 if not path.exists(): return rows
 for number,line in enumerate(path.read_text(encoding='utf-8').splitlines(),1):
  if not line.strip(): continue
  row=json.loads(line); identity=row[id_field]
  if identity in seen: raise ValueError(f'duplicate {id_field} at line {number}: {identity}')
  seen.add(identity); rows.append(row)
 return rows
def shallow_inputs(filename):
 found=[]
 for pattern in (f'*/{filename}',f'*/*/{filename}',f'*/*/*/{filename}'):
  found.extend(path.resolve() for path in INPUT_ROOT.glob(pattern) if path.is_file())
 return sorted(set(found))
def checkpoint_zip():
 temp=Path(str(CHECKPOINT_ZIP)+'.tmp')
 with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4) as z:
  for path in (STATE_JSON,EASY_JSONL,CANDIDATE_JSONL,VINTERN_JSONL,SELECTION_JSON,REPORT_JSON):
   if path.exists(): z.write(path,path.name)
 os.replace(temp,CHECKPOINT_ZIP)

# Restore only known checkpoint members before signature validation. Region crops are rebuilt later from bbox evidence.
checkpoint_inputs=shallow_inputs('ocr_gate_b_checkpoint.zip'); assert len(checkpoint_inputs)<=1,('attach at most one checkpoint',checkpoint_inputs)
if checkpoint_inputs:
 targets={p.name:p for p in (STATE_JSON,EASY_JSONL,CANDIDATE_JSONL,VINTERN_JSONL,SELECTION_JSON,REPORT_JSON)}
 with zipfile.ZipFile(checkpoint_inputs[0]) as z:
  assert z.testzip() is None,'checkpoint ZIP CRC failure'
  names=z.namelist(); assert names and len(names)==len(set(names))
  assert all(name==Path(name).name and name in targets for name in names),('unsafe/unknown checkpoint members',names)
  for name in names:
   payload=z.read(name); target=targets[name]
   if target.exists(): assert target.read_bytes()==payload,('checkpoint conflicts with WORK',name)
   else: target.write_bytes(payload)
 print('CHECKPOINT_RESTORED',checkpoint_inputs[0],names)
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and 'T4' in torch.cuda.get_device_name(0).upper()
print('GPU',torch.cuda.get_device_name(0),'CALIBRATION',CALIBRATION_POLICY['evidence_tier'],CALIBRATION_POLICY['review_rows'])


In [ ]:
# Gate A is a hard prerequisite. Only a verified PASS or explicit deadline override can load models.
gate_reports=shallow_inputs('craft-gate-a-evaluation.json'); gate_artifact_zips=shallow_inputs('ocr_gate_a_emergency_100_artifacts.zip')
if not gate_reports and not gate_artifact_zips:
 print('GATE_A_SHALLOW_LOOKUP_MISS; using one recursive fallback scan')
 gate_reports=sorted(p.resolve() for p in INPUT_ROOT.rglob('craft-gate-a-evaluation.json') if p.is_file()); gate_artifact_zips=sorted(p.resolve() for p in INPUT_ROOT.rglob('ocr_gate_a_emergency_100_artifacts.zip') if p.is_file())
assert len(gate_reports)+len(gate_artifact_zips)==1,('attach exactly one Gate A report or artifact ZIP',[str(p) for p in gate_reports],[str(p) for p in gate_artifact_zips])
if gate_artifact_zips:
 with zipfile.ZipFile(gate_artifact_zips[0]) as z:
  assert z.testzip() is None,'Gate A artifact ZIP CRC failure'; members=[name for name in z.namelist() if Path(name).name=='craft-gate-a-evaluation.json']; assert members==['craft-gate-a-evaluation.json'],('unexpected Gate A report member',members); payload=z.read(members[0])
 GATE_A_REPORT=WORK/'craft-gate-a-evaluation.json'; GATE_A_REPORT.write_bytes(payload); print('GATE_A_REPORT_EXTRACTED',gate_artifact_zips[0],sha256_file(GATE_A_REPORT))
else: GATE_A_REPORT=gate_reports[0]
gate_a=json.loads(GATE_A_REPORT.read_text(encoding='utf-8'))
decision=gate_a.get('decision'); assert decision in ('PASS_THRESHOLD_SELECTED','DEADLINE_OVERRIDE_KEEP_CURRENT') and gate_a.get('gate_b_allowed') is True
assert gate_a.get('policy_sha256')==CRAFT_POLICY_SHA256
config_by_id={c['config_id']:c for c in CRAFT_POLICY['configs']}; selected_id=gate_a.get('selected_config_id'); assert selected_id in config_by_id
SELECTED_CRAFT=config_by_id[selected_id]; assert gate_a.get('selected_thresholds')==SELECTED_CRAFT
selected_metrics=gate_a['metrics'][selected_id]
if decision=='PASS_THRESHOLD_SELECTED':
 assert selected_metrics['eligible'] is True and selected_metrics['region_recall']>=CRAFT_POLICY['min_region_recall'] and selected_metrics['text_frame_recall']>=CRAFT_POLICY['min_text_frame_recall']
else:
 assert selected_id=='recall_current' and selected_metrics['eligible'] is False and gate_a.get('evidence_limitations')==CRAFT_POLICY['evidence_limitations']
assert gate_a['sample']['frames']==100 and gate_a['sample']['videos']==CRAFT_POLICY['sample_video_counts']
RUN_SIGNATURE={'schema_version':2,'gate_a_report_sha256':sha256_file(GATE_A_REPORT),'craft_policy_sha256':CRAFT_POLICY_SHA256,'selected_craft':SELECTED_CRAFT,'router_policy':VINTERN_ROUTER_POLICY,'calibration_policy_sha256':CALIBRATION_POLICY_SHA256,'vintern_model_id':MODEL_ID,'vintern_revision':MODEL_REVISION,'vintern_weight_sha256':MODEL_WEIGHT_SHA256}
if STATE_JSON.exists(): assert json.loads(STATE_JSON.read_text(encoding='utf-8'))==RUN_SIGNATURE,'stale checkpoint signature; attach the matching notebook/checkpoint or use a fresh session'
else: STATE_JSON.write_text(json.dumps(RUN_SIGNATURE,indent=2)+'\n',encoding='utf-8')
print('GATE_A_AUTHORIZED',decision,selected_id,SELECTED_CRAFT,'REPORT_SHA256',RUN_SIGNATURE['gate_a_report_sha256'])


In [ ]:
# Validate exact dev-subset-5 catalog UIDs without a recursive full-Dataset scan unless required.
pattern=re.compile(r'^(s\d+)_(\d+)\.(?:jpg|jpeg|png|webp)$',re.I); catalog_rows=[]; catalog_validation={}
def find_video_dirs(video_id):
 found=[]
 for search in (f'*/keyframes-batch-*/{video_id}',f'*/*/keyframes-batch-*/{video_id}',f'*/*/*/keyframes-batch-*/{video_id}'):
  found.extend(path.resolve() for path in INPUT_ROOT.glob(search) if path.is_dir())
 if not found:
  print('VIDEO_SHALLOW_LOOKUP_MISS',video_id,'using recursive fallback')
  found=[path.resolve() for path in INPUT_ROOT.glob(f'**/keyframes-batch-*/{video_id}') if path.is_dir()]
 return sorted(set(found))
for video_id,expected in DEV_EXPECTED.items():
 matches=find_video_dirs(video_id); assert len(matches)==1,(video_id,[str(p) for p in matches]); entries=[]
 for image_path in sorted(matches[0].iterdir()):
  match=pattern.match(image_path.name)
  if match: entries.append((match.group(1),int(match.group(2)),image_path))
 entries.sort(key=lambda x:(x[0],x[1])); rows=[]
 for local_idx,(shot_id,filename_index,image_path) in enumerate(entries): rows.append({'video_id':video_id,'shot_id':shot_id,'local_idx':local_idx,'filename_index':filename_index,'keyframe_uid':make_uid(video_id,shot_id,local_idx),'source_image':image_path.relative_to(INPUT_ROOT).as_posix(),'source_path':str(image_path)})
 uids=[r['keyframe_uid'] for r in rows]; assert len(rows)==expected['count'] and len(uids)==len(set(uids)); assert uid_set_sha256(uids)==expected['uid_set_sha256']
 catalog_validation[video_id]={'count':len(rows),'uid_set_sha256':uid_set_sha256(uids),'directory':str(matches[0])}; catalog_rows.extend(rows)
catalog_rows.sort(key=lambda r:(r['video_id'],r['shot_id'],r['local_idx'])); assert len(catalog_rows)==4164 and len({r['keyframe_uid'] for r in catalog_rows})==4164
print('DEV_TOTAL',len(catalog_rows),json.dumps(catalog_validation,indent=2))


In [ ]:
# Pinned CRAFT + latin_g2, full EasyOCR pass with UID resume and the Gate A-selected detector threshold.
import easyocr
for key,spec in EASYOCR_WEIGHTS.items():
 archive=WORK/f'{key}.zip'; weight=MODEL_DIR/spec['weight_name']
 if not archive.exists() or sha256_file(archive)!=spec['zip_sha256']: urllib.request.urlretrieve(spec['url'],archive)
 assert sha256_file(archive)==spec['zip_sha256']
 if not weight.exists() or sha256_file(weight)!=spec['weight_sha256']:
  with zipfile.ZipFile(archive) as z:
   member=next(name for name in z.namelist() if name.endswith(spec['weight_name']))
   with z.open(member) as source,weight.open('wb') as target: shutil.copyfileobj(source,target)
 assert sha256_file(weight)==spec['weight_sha256']
reader=easyocr.Reader(['vi','en'],gpu='cuda:0',model_storage_directory=str(MODEL_DIR),download_enabled=False,detector=True,recognizer=True,verbose=True)
def quad_flat(points): return [float(value) for point in points for value in point]
def normalize_quad(points,width,height): return [max(0.,min(1.,float(v)/(width if i%2==0 else height))) for i,v in enumerate(quad_flat(points))]
def vi_marks(text):
 return any(c in 'ăâđêôơưĂÂĐÊÔƠƯ' or any(mark in unicodedata.normalize('NFD',c) for mark in '\u0300\u0301\u0303\u0309\u0323') for c in text)
def ascii_word(text): return bool(re.search(r'[A-Za-z]{3,}',text))
def process_frame(row):
 started=time.perf_counter(); image=cv2.imread(row['source_path'],cv2.IMREAD_COLOR)
 if image is None: return {**row,'schema_version':1,'status':'error','error':'cv2_imread_failed','regions':[]}
 height,width=image.shape[:2]
 try:
  detect_started=time.perf_counter(); horizontal,free=reader.detect(image,min_size=10,text_threshold=SELECTED_CRAFT['text_threshold'],low_text=SELECTED_CRAFT['low_text'],link_threshold=SELECTED_CRAFT['link_threshold'],canvas_size=2560,mag_ratio=1.0,slope_ths=0.1,ycenter_ths=0.5,height_ths=0.5,width_ths=0.5,add_margin=0.1,reformat=True); detect_seconds=time.perf_counter()-detect_started
  h0=horizontal[0] if horizontal else []; f0=free[0] if free else []; count=len(h0)+len(f0)
  if count==0: return {**row,'schema_version':1,'status':'no_text','image_width':width,'image_height':height,'detected_region_count':0,'detect_seconds':detect_seconds,'recognize_seconds':0.,'latency_seconds':time.perf_counter()-started,'error':None,'regions':[]}
  grey=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY); recognize_started=time.perf_counter(); results=reader.recognize(grey,h0,f0,decoder='greedy',beamWidth=5,batch_size=1,workers=0,detail=1,paragraph=False,contrast_ths=0.1,adjust_contrast=0.5,filter_ths=0.003,reformat=False); recognize_seconds=time.perf_counter()-recognize_started
  if len(results)!=count: raise RuntimeError(f'recognizer returned {len(results)} rows for {count} regions')
  parsed=[]
  for index,(points,text_value,confidence_value) in enumerate(results):
   points=np.asarray(points,dtype=np.float32).reshape(4,2); text_value=str(text_value).strip(); confidence=float(max(0.,min(1.,confidence_value))); x1=max(0,int(math.floor(points[:,0].min()))-4); y1=max(0,int(math.floor(points[:,1].min()))-4); x2=min(width,int(math.ceil(points[:,0].max()))+4); y2=min(height,int(math.ceil(points[:,1].max()))+4); region_id=stable_id(row['keyframe_uid'],index,';'.join(f'{v:.2f}' for v in quad_flat(points))); crop=CROP_DIR/f'{region_id}.jpg'
   if not crop.exists() and x2>x1 and y2>y1: assert cv2.imwrite(str(crop),image[y1:y2,x1:x2])
   parsed.append({'region_id':region_id,'bbox_px':quad_flat(points),'bbox_normalized':normalize_quad(points,width,height),'crop_path':str(crop),'crop_width':x2-x1,'crop_height':y2-y1,'easyocr_text':text_value,'easyocr_confidence':confidence,'has_vi_marks':vi_marks(text_value),'has_ascii_word':ascii_word(text_value)})
  return {**row,'schema_version':1,'status':'text_detected','image_width':width,'image_height':height,'detected_region_count':count,'detect_seconds':detect_seconds,'recognize_seconds':recognize_seconds,'latency_seconds':time.perf_counter()-started,'error':None,'regions':parsed}
 except Exception as exc: return {**row,'schema_version':1,'status':'error','image_width':width,'image_height':height,'detected_region_count':None,'detect_seconds':None,'recognize_seconds':None,'latency_seconds':time.perf_counter()-started,'error':f'{type(exc).__name__}: {str(exc)[:500]}','regions':[]}
existing=load_jsonl(EASY_JSONL,'keyframe_uid'); done={r['keyframe_uid'] for r in existing}; print('EASYOCR_RESUME',len(done),'/',len(catalog_rows))
completed=len(done)
for row in catalog_rows:
 if row['keyframe_uid'] in done: continue
 append_jsonl(EASY_JSONL,process_frame(row)); done.add(row['keyframe_uid']); completed+=1
 if completed%25==0 or completed==len(catalog_rows): print('EASYOCR_GATE_B_PROGRESS',completed,'/',len(catalog_rows))
 if completed%250==0: checkpoint_zip()
easy_rows=load_jsonl(EASY_JSONL,'keyframe_uid'); assert len(easy_rows)==4164 and {r['keyframe_uid'] for r in easy_rows}=={r['keyframe_uid'] for r in catalog_rows}; assert not [r for r in easy_rows if r['status']=='error']
current_by_uid={r['keyframe_uid']:r for r in catalog_rows}
for frame in easy_rows:
 current=current_by_uid[frame['keyframe_uid']]; frame['source_path']=current['source_path']; frame['source_image']=current['source_image']
rewrite_jsonl(EASY_JSONL,easy_rows)
del reader; gc.collect(); torch.cuda.empty_cache(); checkpoint_zip(); print('EASYOCR_STATUS',Counter(r['status'] for r in easy_rows))


In [ ]:
# Router v2 is region-local; it never propagates mixed state across the frame.
ALLOWED=set('.,:;!?%+-/()[]{}\'"&@#_\\|~`=<>₫$€£¥…–—')
def noise_ratio(text):
 visible=[c for c in text if not c.isspace()]
 if not visible: return 0.
 def noisy(c): return not (c.isalnum() or c in ALLOWED or unicodedata.category(c).startswith(('L','N','P','S','M')))
 return sum(noisy(c) for c in visible)/len(visible)
def route(region):
 text=str(region.get('easyocr_text') or '').strip(); confidence=float(region.get('easyocr_confidence') or 0.); reasons=[]
 if not text: reasons.append('empty_text')
 elif confidence<0.4: reasons.append('confidence_lt_0_40')
 elif confidence<0.6:
  if region.get('has_vi_marks') and region.get('has_ascii_word'): reasons.append('region_mixed_0_40_to_0_60')
  if '?' in text or '�' in text: reasons.append('ambiguous_glyph_0_40_to_0_60')
  visible=len(''.join(text.split()))
  if visible>=3 and noise_ratio(text)>=0.34: reasons.append('noisy_text_0_40_to_0_60')
 return reasons
candidates=[]
for frame in easy_rows:
 for region in frame.get('regions',[]):
  reasons=route(region)
  if reasons: candidates.append({'schema_version':1,'candidate_id':region['region_id'],'video_id':frame['video_id'],'keyframe_uid':frame['keyframe_uid'],'shot_id':frame['shot_id'],'local_idx':frame['local_idx'],'source_image':frame['source_image'],'source_path':frame['source_path'],'bbox_px':region['bbox_px'],'crop_path':str(CROP_DIR/f"{region['region_id']}.jpg"),'crop_width':region['crop_width'],'crop_height':region['crop_height'],'easyocr_text':region['easyocr_text'],'easyocr_confidence':region['easyocr_confidence'],'router_v2_reasons':reasons})
candidates.sort(key=lambda r:(r['video_id'],r['keyframe_uid'],r['candidate_id'])); assert len({r['candidate_id'] for r in candidates})==len(candidates)
fraction=len(candidates)/max(1,sum(len(f.get('regions',[])) for f in easy_rows)); assert fraction<=VINTERN_ROUTER_POLICY['max_candidate_fraction'],(len(candidates),fraction); rewrite_jsonl(CANDIDATE_JSONL,candidates)

# A restored JSONL checkpoint intentionally excludes bulky crops. Rebuild only missing candidate crops from pinned bbox/source evidence.
missing_by_source=defaultdict(list)
for candidate in candidates:
 if not Path(candidate['crop_path']).is_file(): missing_by_source[candidate['source_path']].append(candidate)
for source_index,(source_path,rows) in enumerate(sorted(missing_by_source.items()),1):
 image=cv2.imread(source_path,cv2.IMREAD_COLOR); assert image is not None,('cannot rebuild crop',source_path); height,width=image.shape[:2]
 for candidate in rows:
  flat=candidate['bbox_px']; assert len(flat)==8; xs=flat[0::2]; ys=flat[1::2]; x1=max(0,int(math.floor(min(xs)))-4); y1=max(0,int(math.floor(min(ys)))-4); x2=min(width,int(math.ceil(max(xs)))+4); y2=min(height,int(math.ceil(max(ys)))+4); assert x2>x1 and y2>y1; assert cv2.imwrite(candidate['crop_path'],image[y1:y2,x1:x2])
 if source_index%100==0 or source_index==len(missing_by_source): print('CROP_RESTORE_PROGRESS',source_index,'/',len(missing_by_source))
assert all(Path(candidate['crop_path']).is_file() for candidate in candidates)
prior=load_jsonl(VINTERN_JSONL,'candidate_id'); assert {r['candidate_id'] for r in prior}<={r['candidate_id'] for r in candidates},'stale Vintern checkpoint candidate IDs'
checkpoint_zip(); print('ROUTER_V2',{'regions':sum(len(f.get('regions',[])) for f in easy_rows),'candidates':len(candidates),'fraction':fraction,'reasons':dict(Counter(reason for c in candidates for reason in c['router_v2_reasons'])),'restored_crops':sum(len(rows) for rows in missing_by_source.values())})


In [ ]:
# Load and verify the exact official Vintern FP16 revision only when checkpoint results remain unfinished.
prior=load_jsonl(VINTERN_JSONL,'candidate_id'); pending_candidate_ids={r['candidate_id'] for r in candidates}-{r['candidate_id'] for r in prior}; model=tokenizer=None
if not pending_candidate_ids:
 print('VINTERN_RESUME_COMPLETE',len(prior),'/',len(candidates),'model load skipped')
else:
 from huggingface_hub import snapshot_download
 from transformers import AutoModel,AutoTokenizer
 snapshot_dir=Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION)); weight_path=snapshot_dir/'model.safetensors'; assert weight_path.stat().st_size==MODEL_WEIGHT_BYTES and sha256_file(weight_path)==MODEL_WEIGHT_SHA256
 for filename,expected in MODEL_REQUIRED_GIT_OIDS.items(): assert git_blob_oid(snapshot_dir/filename)==expected,(filename,git_blob_oid(snapshot_dir/filename),expected)
 runtime=WORK/'vintern-local-runtime'
 if runtime.exists(): shutil.rmtree(runtime)
 runtime.mkdir()
 for source in snapshot_dir.iterdir():
  target=runtime/source.name
  if source.name=='config.json':
   config=json.loads(source.read_text(encoding='utf-8')); config['auto_map']={k:v.split('--',1)[-1] for k,v in config['auto_map'].items()}; target.write_text(json.dumps(config,indent=2),encoding='utf-8')
  elif source.is_file(): target.symlink_to(source.resolve())
 torch.cuda.empty_cache(); tokenizer=AutoTokenizer.from_pretrained(str(runtime),trust_remote_code=True,use_fast=False,local_files_only=True); model=AutoModel.from_pretrained(str(runtime),torch_dtype=torch.float16,low_cpu_mem_usage=True,trust_remote_code=True,local_files_only=True,use_flash_attn=False).eval().to('cuda:0')
 assert all(p.dtype==torch.float16 for p in model.parameters() if p.is_floating_point()); print('VINTERN_READY',len(pending_candidate_ids),'pending',torch.cuda.memory_allocated(0)/2**20,'MiB')


In [ ]:
# Same-pass Vintern inference. Internal signals are derived from this result; log-prob is unavailable from model.chat and remains null.
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
transform=T.Compose([T.Lambda(lambda image:image.convert('RGB')),T.Resize((448,448),interpolation=InterpolationMode.BICUBIC),T.ToTensor(),T.Normalize(mean=(0.485,0.456,0.406),std=(0.229,0.224,0.225))])
QUESTION='<image>\nChép lại nguyên văn toàn bộ chữ nhìn thấy. Không sửa chính tả, không suy đoán phần bị che. Chỉ trả về văn bản.'
GENERATION={'max_new_tokens':96,'do_sample':False,'num_beams':1,'eos_token_id':tokenizer.eos_token_id,'pad_token_id':tokenizer.eos_token_id}
PROMPT_MARKERS=('i do not understand','this is a blurry image','the image is blurry','the text is blurry','chữ bị che khuất','chữ trong ảnh','nội dung trong ảnh','không thể đọc','không đọc được','từ đó tôi sẽ','hãy chép lại','chép lại nguyên văn')
def load_pixels(path): return transform(Image.open(path).convert('RGB')).unsqueeze(0).to('cuda:0',dtype=torch.float16)
def guard_reasons(easy,text):
 candidate=text.strip()
 if not candidate: return ['empty_output']
 reasons=[]; normalized=' '.join(candidate.casefold().split())
 if any(marker in normalized for marker in PROMPT_MARKERS): reasons.append('prompt_or_explanation_leak')
 easy_len=len(''.join(easy.split())); candidate_len=len(''.join(candidate.split()))
 if candidate_len>max(96,easy_len*8+48): reasons.append('gross_length_expansion')
 if noise_ratio(candidate)>=0.5 and candidate_len>=4: reasons.append('noisy_output')
 return reasons
prior=load_jsonl(VINTERN_JSONL,'candidate_id'); done={r['candidate_id'] for r in prior}; completed=len(done); print('VINTERN_RESUME',completed,'/',len(candidates)); warm=next((r for r in candidates if r['candidate_id'] not in done),None)
if warm:
 pixels=load_pixels(warm['crop_path'])
 with torch.inference_mode(): model.chat(tokenizer,pixels,QUESTION,GENERATION)
 del pixels; torch.cuda.synchronize()
for index,candidate in enumerate(candidates,1):
 if candidate['candidate_id'] in done: continue
 started=time.perf_counter()
 try:
  pixels=load_pixels(candidate['crop_path']); torch.cuda.synchronize(); infer=time.perf_counter()
  with torch.inference_mode(): text=str(model.chat(tokenizer,pixels,QUESTION,GENERATION)).strip()
  torch.cuda.synchronize(); inference_seconds=time.perf_counter()-infer; del pixels; easy_len=len(''.join(candidate['easyocr_text'].split())); output_len=len(''.join(text.split())); limit=max(96,easy_len*8+48); guards=guard_reasons(candidate['easyocr_text'],text)
  record={**candidate,'status':'success' if text else 'empty','vintern_text':text,'inference_seconds':inference_seconds,'total_seconds':time.perf_counter()-started,'output_length':output_len,'guard_length_limit':limit,'guard_margin_ratio':(limit-output_len)/limit,'mean_token_logprob':None,'logprob_available':False,'guard_rejection_reasons':guards,'error':None}
 except Exception as exc:
  if isinstance(exc,torch.cuda.OutOfMemoryError): torch.cuda.empty_cache()
  record={**candidate,'status':'error','vintern_text':'','inference_seconds':None,'total_seconds':time.perf_counter()-started,'output_length':0,'guard_length_limit':max(96,len(''.join(candidate['easyocr_text'].split()))*8+48),'guard_margin_ratio':1.0,'mean_token_logprob':None,'logprob_available':False,'guard_rejection_reasons':['runtime_error'],'error':f'{type(exc).__name__}: {str(exc)[:500]}'}
 append_jsonl(VINTERN_JSONL,record); done.add(candidate['candidate_id']); completed+=1
 if completed%25==0 or completed==len(candidates): print('VINTERN_GATE_B_PROGRESS',completed,'/',len(candidates))
 if completed%250==0: checkpoint_zip()
vintern_rows=load_jsonl(VINTERN_JSONL,'candidate_id'); assert {r['candidate_id'] for r in vintern_rows}=={r['candidate_id'] for r in candidates}; assert not [r for r in vintern_rows if r['status']=='error']
if model is not None: del model,tokenizer
gc.collect(); torch.cuda.empty_cache(); checkpoint_zip(); print('VINTERN_STATUS',Counter(r['status'] for r in vintern_rows),Counter(reason for r in vintern_rows for reason in r['guard_rejection_reasons']))


In [ ]:
# Emergency human review: exactly 100 candidates, 20/video, one candidate per frame, deterministic stratified selection.
STRATA=('empty_text','confidence_lt_0_30','confidence_0_30_to_0_40','mixed_0_40_to_0_60','glyph_or_noise_0_40_to_0_60')
def review_stratum(row):
 text=str(row.get('easyocr_text') or '').strip(); confidence=float(row.get('easyocr_confidence') or 0.); reasons=set(row.get('router_v2_reasons') or [])
 if not text: return 'empty_text'
 if confidence<0.30: return 'confidence_lt_0_30'
 if confidence<0.40: return 'confidence_0_30_to_0_40'
 if 'region_mixed_0_40_to_0_60' in reasons: return 'mixed_0_40_to_0_60'
 return 'glyph_or_noise_0_40_to_0_60'
def selection_rank(row): return hashlib.sha256((CALIBRATION_POLICY['review_selection_seed']+'|'+row['candidate_id']).encode()).hexdigest()
eligible=[dict(row,calibration_stratum=review_stratum(row)) for row in vintern_rows if row['status'] in ('success','empty')]
review=[]; per_video_strata={}
for video_id in CRAFT_POLICY['video_ids']:
 rows=sorted((row for row in eligible if row['video_id']==video_id),key=selection_rank); selected=[]; used_frames=set()
 for stratum in STRATA:
  taken=0
  for row in (candidate for candidate in rows if candidate['calibration_stratum']==stratum):
   if row['keyframe_uid'] in used_frames: continue
   selected.append(row); used_frames.add(row['keyframe_uid']); taken+=1
   if taken==CALIBRATION_POLICY['review_rows_per_stratum_per_video']: break
 for row in rows:
  if len(selected)==CALIBRATION_POLICY['review_rows_per_video']: break
  if row['keyframe_uid'] in used_frames: continue
  selected.append(row); used_frames.add(row['keyframe_uid'])
 assert len(selected)==CALIBRATION_POLICY['review_rows_per_video'],(video_id,len(selected)); review.extend(selected); per_video_strata[video_id]=dict(Counter(row['calibration_stratum'] for row in selected))
review.sort(key=lambda r:(r['video_id'],selection_rank(r))); assert len(review)==CALIBRATION_POLICY['review_rows'] and len({r['keyframe_uid'] for r in review})==CALIBRATION_POLICY['review_rows'] and Counter(r['video_id'] for r in review)==Counter({v:CALIBRATION_POLICY['review_rows_per_video'] for v in CRAFT_POLICY['video_ids']})
selection_evidence=[{'candidate_id':row['candidate_id'],'keyframe_uid':row['keyframe_uid'],'video_id':row['video_id'],'stratum':row['calibration_stratum']} for row in review]
selection_sha256=hashlib.sha256(canonical_json(selection_evidence)).hexdigest(); selection_manifest={'schema_version':1,'created_utc':datetime.now(timezone.utc).isoformat(),'selection_sha256':selection_sha256,'calibration_policy_sha256':CALIBRATION_POLICY_SHA256,'selection_seed':CALIBRATION_POLICY['review_selection_seed'],'rows':len(review),'distinct_frames':len({r['keyframe_uid'] for r in review}),'rows_per_video':dict(Counter(r['video_id'] for r in review)),'strata':dict(Counter(r['calibration_stratum'] for r in review)),'strata_per_video':per_video_strata,'selected':selection_evidence}
SELECTION_JSON.write_text(json.dumps(selection_manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
font=ImageFont.load_default(); csv_rows=[]
for index,row in enumerate(review,1):
 source=Image.open(row['source_path']).convert('RGB'); flat=row['bbox_px']; points=[(flat[i],flat[i+1]) for i in range(0,8,2)]; context=source.copy(); draw=ImageDraw.Draw(context); draw.line(points+[points[0]],fill=(255,40,40),width=max(3,source.width//400)); context.thumbnail((900,500),Image.Resampling.LANCZOS)
 crop=Image.open(row['crop_path']).convert('RGB'); crop.thumbnail((900,220),Image.Resampling.LANCZOS); sheet=Image.new('RGB',(920,790),'white'); sheet.paste(context,((920-context.width)//2,35)); sheet.paste(crop,((920-crop.width)//2,550)); label=f'#{index} {row["video_id"]} [{row["calibration_stratum"]}]\nEasyOCR: {row["easyocr_text"][:100]} ({row["easyocr_confidence"]:.3f})\nVintern: {row["vintern_text"][:120]}'; ImageDraw.Draw(sheet).multiline_text((10,8),label,fill='black',font=font,spacing=2)
 image_name=f'{index:03d}_{row["video_id"]}_{row["candidate_id"]}.jpg'; sheet.save(REVIEW_DIR/image_name,quality=90,optimize=True)
 csv_rows.append({'sample_index':index,'image_file':f'review-images/{image_name}','video_id':row['video_id'],'keyframe_uid':row['keyframe_uid'],'candidate_id':row['candidate_id'],'region_id':row['candidate_id'],'calibration_stratum':row['calibration_stratum'],'selection_sha256':selection_sha256,'easyocr_text':row['easyocr_text'],'easyocr_confidence':row['easyocr_confidence'],'vintern_text':row['vintern_text'],'output_length':row['output_length'],'guard_margin_ratio':row['guard_margin_ratio'],'mean_token_logprob':'','label_status':'','ground_truth_is_empty':'','human_text':'','annotator':'','notes':''})
with REVIEW_CSV.open('w',encoding='utf-8-sig',newline='') as h:
 writer=csv.DictWriter(h,fieldnames=list(csv_rows[0])); writer.writeheader(); writer.writerows(csv_rows)
instructions=WORK/'VINTERN_GROUND_TRUTH_INSTRUCTIONS.txt'; instructions.write_text('Label all 100 rows. Inspect context, red region, and enlarged crop. For a usable row set label_status=labeled. If the region is detector noise with no real text, set ground_truth_is_empty=yes and leave human_text empty; otherwise set ground_truth_is_empty=no and transcribe exact visible text. If a human truly cannot read it, set label_status=exclude_unreadable and STOP before calibration: select a deterministic replacement from this same checkpoint; never invent text and never rerun Vintern. Fill annotator. Do not change IDs, model signals, stratum, or selection_sha256. This is emergency single-annotator-100 evidence, not a standard 300-frame calibration PASS. Buckets with support below 20 cannot override EasyOCR and become Gemini residual.\n',encoding='utf-8')
easy_seconds=sum(float(r.get('latency_seconds') or 0) for r in easy_rows); vintern_seconds=sum(float(r.get('inference_seconds') or 0) for r in vintern_rows); total_regions=sum(len(r.get('regions',[])) for r in easy_rows); fanout=len(candidates)/max(1,total_regions)
report={'schema_version':2,'created_utc':datetime.now(timezone.utc).isoformat(),'decision':'PENDING_EMERGENCY_100_VINTERN_HUMAN_GROUND_TRUTH','gate_a':{'report_sha256':sha256_file(GATE_A_REPORT),'selected_craft':SELECTED_CRAFT},'scope':'full_dev_subset_5_no_gemini','catalog':{'frames':4164,'videos':catalog_validation},'easyocr':{'status':dict(Counter(r['status'] for r in easy_rows)),'regions':total_regions,'seconds':easy_seconds,'frames_per_second':4164/easy_seconds},'router_v2':{'candidates':len(candidates),'candidate_fraction':fanout,'max_fraction':VINTERN_ROUTER_POLICY['max_candidate_fraction']},'vintern':{'results':len(vintern_rows),'status':dict(Counter(r['status'] for r in vintern_rows)),'guard_rejections':sum(bool(r['guard_rejection_reasons']) for r in vintern_rows),'seconds':vintern_seconds,'candidates_per_second':len(vintern_rows)/vintern_seconds},'calibration_review':{'evidence_tier':CALIBRATION_POLICY['evidence_tier'],'rows':len(review),'distinct_frames':len({r['keyframe_uid'] for r in review}),'required_labeled_distinct_frames':CALIBRATION_POLICY['min_ground_truth_frames'],'videos':dict(Counter(r['video_id'] for r in review)),'strata':dict(Counter(r['calibration_stratum'] for r in review)),'strata_per_video':per_video_strata,'selection_sha256':selection_sha256,'calibration_policy_sha256':CALIBRATION_POLICY_SHA256,'global_bucket_override_allowed':False,'human_text_status':'blank_pending'},'full_catalog_eta_single_t4_dev_extrapolation_only':{'easyocr_hours':293336/(4164/easy_seconds)/3600,'vintern_candidates_estimated':round(len(candidates)/4164*293336),'vintern_hours':(len(candidates)/4164*293336)/(len(vintern_rows)/vintern_seconds)/3600},'model_calls':{'gemini':0},'limitations':['Emergency single-annotator calibration uses 100 labeled regions/frames, not the standard 300-frame evidence tier.','A calibration bucket needs support >=20; global backoff is report-only and cannot authorize an override.']}
REPORT_JSON.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8'); checkpoint_zip()
with zipfile.ZipFile(REVIEW_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for path in (REVIEW_CSV,REPORT_JSON,SELECTION_JSON,instructions): z.write(path,path.name)
 for path in sorted(REVIEW_DIR.glob('*.jpg')): z.write(path,'review-images/'+path.name)
print(json.dumps(report,ensure_ascii=False,indent=2)); print('DOWNLOAD_CHECKPOINT',CHECKPOINT_ZIP,CHECKPOINT_ZIP.stat().st_size,sha256_file(CHECKPOINT_ZIP)); print('DOWNLOAD_GROUND_TRUTH_REVIEW',REVIEW_ZIP,REVIEW_ZIP.stat().st_size,sha256_file(REVIEW_ZIP))
